# WANDS: MUVERA fidelity and serving costs

Continue from the **completed DenseOn/LateOn WANDS run**. Preserve its queries, human judgments, model revisions and reference scores. Document embeddings must be re-encoded once because the quality notebook did not retain them.

Compare exact dense and LateOn search, BM25, the shared-pool cross-encoder pipeline, and MUVERA flat/HNSW + live MaxSim reranking. Default sweep: 4096/8192 FDE dimensions, 100–10,000 candidates, fidelity@10 on all queries, latency on 30 seeded queries × 3 repeats.

**Device scope:** model encoding on GPU; all search and MaxSim reranking on CPU (2 threads). NumPy FDE reference, not Google's optimized C++/DiskANN/PQ stack. No all-GPU, IVF or quantization claims.

Select a GPU runtime. A high-RAM runtime is useful for token arrays. Work runs on local disk; checkpoints are copied to Drive after each phase, including ordinary failures. A runtime reset can lose the currently running phase. Do not run another benchmark simultaneously.


In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

def run_logged(command, *, cwd=None, log_path):
    """Forward child stdout/stderr through notebook output and keep the failure tail."""
    import collections
    import subprocess
    import sys
    from pathlib import Path

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = collections.deque(maxlen=80)
    print(f'Python: {sys.version.split()[0]} | executable: {sys.executable}', flush=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n--- New invocation ---\n')
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                              errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    tail.append(line)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if returncode:
        raise RuntimeError(
            f'Command exited with status {returncode}. Full log: {log_path}\n'
            + ''.join(tail))
    return returncode

drive.mount('/content/drive')
REPO = Path('/content/ras-wands-systems')
BRANCH = 'codex/colbert-muvera-baselines'
LOGS = Path('/content/drive/MyDrive/ras_wands_systems_logs')
if not REPO.exists():
    run_logged(['git', 'clone', '--branch', BRANCH, '--single-branch',
                'https://github.com/hanialshater/ras.git', str(REPO)],
               log_path=LOGS / 'setup.log')
else:
    run_logged(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH],
               log_path=LOGS / 'setup.log')
print(subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True))
run_logged([sys.executable, '-m', 'pip', 'install', '-e',
            str(REPO) + '[dev,wands,wands-systems]'],
           log_path=LOGS / 'install.log')


In [ ]:
import os
os.chdir(REPO)
os.environ.update(PYTHONPATH=str(REPO / 'src') + ':' + str(REPO), USE_TF='0', USE_FLAX='0',
                  PYLATE_SCORES_BACKEND='torch', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
run_logged([sys.executable, '-c', 'import torch, pylate, faiss, psutil, ir_measures; assert torch.cuda.is_available(), "Select GPU runtime"'], cwd=REPO, log_path=LOGS / 'environment.log')
run_logged([sys.executable, '-m', 'pytest', '-q', 'tests/test_wands_systems.py', 'tests/test_wands_comparison.py', 'tests/test_late_interaction.py'], cwd=REPO, log_path=LOGS / 'tests.log')


## Locate the completed quality run

Change `REFERENCE_DRIVE` if you used a different folder. Use a **new** `RUN_NAME` if you change dimensions, budgets, models, code, hardware or timing settings. Existing quality results remain the reference.


In [ ]:
import json, shutil
REFERENCE_DRIVE = Path('/content/drive/MyDrive/ras_wands_denseon_lateon_seed7_v1')
RUN_NAME = 'ras_wands_muvera_cost_seed7_v1'
REFERENCE = Path('/content/wands_systems_reference')
RUN = Path('/content') / RUN_NAME
BACKUP = Path('/content/drive/MyDrive') / RUN_NAME
DIMENSIONS = [4096, 8192]
CANDIDATES = [100, 500, 1000, 2000, 5000, 10000]

assert json.loads((REFERENCE_DRIVE / 'complete.json').read_text())['status'] == 'complete'
def mirror(source, destination):
    destination.mkdir(parents=True, exist_ok=True)
    copied = 0
    for path in source.rglob('*'):
        if not path.is_file() or '.tmp' in path.name or path.name.endswith('.copying'):
            continue
        target = destination / path.relative_to(source)
        if target.exists() and target.stat().st_size == path.stat().st_size and abs(target.stat().st_mtime-path.stat().st_mtime) < 2:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        temp = target.with_suffix(target.suffix + '.copying')
        shutil.copy2(path, temp)
        temp.replace(target)
        copied += 1
    print(f'Copied {copied} checkpoint files to {destination}', flush=True)

mirror(REFERENCE_DRIVE, REFERENCE)
if BACKUP.exists():
    mirror(BACKUP, RUN)
RUN.mkdir(parents=True, exist_ok=True)
BASE = [sys.executable, '-u', '-m', 'experiments.wands_systems',
        '--reference-dir', str(REFERENCE), '--output-dir', str(RUN),
        '--dimensions', *map(str, DIMENSIONS), '--candidates', *map(str, CANDIDATES),
        '--k', '10', '--seed', '7', '--threads', '2', '--timing-queries', '30',
        '--timing-repeats', '3', '--warmup', '2', '--device', 'cuda']
def phase(name, *extra):
    try:
        run_logged(BASE + ['--phase', name, *map(str, extra)], cwd=REPO,
                   log_path=LOGS / ('_'.join([name, *map(str, extra)]).replace('--', '') + '.log'))
    finally:
        mirror(RUN, BACKUP)  # No background Drive sync during measured requests.
phase('prepare')


## Encode documents once and verify reference parity

Encoding resumes from completed shards. After encoding, five seeded queries are scored against **every product** and checked against the completed reference, with numeric tolerance and top-k overlap recorded. Failure blocks subsequent phases.


In [ ]:
for arm in ['dense', 'colbert']:
    phase('encode', '--arm', arm)
phase('parity')
print((RUN / 'parity.json').read_text())


## Build FDEs and measure approximation fidelity

FDE flat search isolates representation loss. HNSW adds ANN search error. Both rerank candidates by exact LateOn reference scores for this offline quality calculation. Timed serving below recomputes scores from token vectors.


In [ ]:
for dimension in DIMENSIONS:
    phase('transform', '--dimension', dimension)
    for backend in ['flat', 'hnsw']:
        phase('index', '--arm', backend, '--dimension', dimension)
        phase('fidelity', '--arm', backend, '--dimension', dimension)


## Measure serving latency and memory in isolated workers

Each call starts a fresh process with only one serving pipeline. Live query encoding, FDE creation, CPU retrieval, reranking and selection are timed separately. Cross-encoder timing includes live dense + BM25 union generation, with a check against the original candidate IDs.

These are warmed, sequential measurements at concurrency 1. Token arrays are memory mapped; operating-system paging remains part of the observed cost. p99 is exploratory at this sample size. Model loading is recorded separately; whole-process RAM and PyTorch CUDA allocator peaks are distinct.


In [ ]:
for arm in ['dense', 'colbert', 'bm25', 'ce']:
    phase('latency', '--arm', arm)
for dimension in DIMENSIONS:
    for backend in ['flat', 'hnsw']:
        phase('latency', '--arm', backend, '--dimension', dimension)
phase('report')
(RUN / 'complete.json').write_text(json.dumps({'status': 'complete'}))
mirror(RUN, BACKUP)


## Results

Read relevance separately from fidelity. Fidelity targets are selected on this run's mean; they are not independent guarantees. Storage counts each serving pipeline once, including full-precision tokens for MaxSim and FDE vectors inside the FAISS index. Offline score/query caches and checkpoints are excluded. Build components may span resumed sessions; inspect their provenance.


In [ ]:
import pandas as pd
from IPython.display import display
print('Original quality reference')
display(pd.read_csv(REFERENCE / 'quality.csv'))
for filename in ['fidelity.csv', 'muvera_quality.csv', 'latency_at_fidelity.csv', 'latency.csv', 'memory.csv', 'storage_build.csv']:
    print(filename)
    display(pd.read_csv(RUN / filename))
print((RUN / 'scope.json').read_text())


In [ ]:
# Small export: reports, settings and component timing; excludes large arrays/indexes.
import zipfile
from google.colab import files
archive = Path('/content') / (RUN_NAME + '_reports.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in RUN.rglob('*'):
        if path.is_file() and path.suffix in ['.csv', '.json'] and 'rankings' not in path.name and path.name != 'serving.json' and not path.name.startswith('shard_'):
            z.write(path, path.relative_to(RUN))
files.download(str(archive))
